# PHẦN I — CHUẨN BỊ SERVER VẬT LÝ

Notebook này dùng để lưu code và chú thích khi chuẩn bị **3 server vật lý A, B, C** cho mô hình OpenStack / Kolla-Ansible / Ceph.

## Cách dùng

- Chạy lần lượt trên **Server A trước**.
- Sau khi kiểm tra ổn, lặp lại tương tự trên **Server B** và **Server C**.
- Với các bước có nguy cơ mất dữ liệu, đặc biệt là **LVM / phân vùng / Ceph OSD**, phải đọc kỹ chú thích trước khi chạy.
- Các cell code bên dưới là lệnh shell. Nếu chạy trong Jupyter, nên dùng kernel Bash hoặc copy trực tiếp vào terminal SSH.

## Quy ước server

| Server | Vai trò | Ghi chú |
|---|---|---|
| Server A | Physical node A | Hypervisor + Compute bare metal + Ceph OSD |
| Server B | Physical node B | Hypervisor + Compute bare metal + Ceph OSD |
| Server C | Physical node C | Hypervisor + Compute bare metal + Ceph OSD |

## BƯỚC 1 — Cập nhật hệ thống

Thực hiện trên cả 3 server.

Mục tiêu:
- Cập nhật package list.
- Nâng cấp toàn bộ hệ điều hành.
- Reboot để kernel / service mới được áp dụng.

In [ ]:
# Server A/B/C — update OS
sudo apt update && sudo apt full-upgrade -y
sudo reboot

Sau khi server reboot xong, SSH lại vào server và kiểm tra kernel version.

In [ ]:
uname -r

## BƯỚC 2 — Cài packages nền tảng

Mục tiêu:
- Cài KVM/libvirt để tạo và quản lý VM.
- Cài Open vSwitch để chuẩn bị bridge/network cho OpenStack.
- Cài LVM/thin provisioning để tạo pool lưu VM local.
- Cài các công cụ hệ thống cần cho debug, network, disk, firewall.

In [ ]:
sudo apt install -y \
  qemu-kvm libvirt-daemon-system libvirt-clients \
  virtinst libguestfs-tools cloud-image-utils genisoimage \
  openvswitch-switch openvswitch-common \
  lvm2 thin-provisioning-tools gdisk parted \
  bridge-utils dnsmasq \
  iptables iptables-persistent netfilter-persistent \
  curl wget git vim htop iotop net-tools tcpdump

## BƯỚC 3 — Cấu hình user và service

Mục tiêu:
- Cho user hiện tại quyền dùng libvirt/KVM.
- Bật libvirtd.
- Xóa network NAT default của libvirt để tránh conflict với mạng lab/OpenStack.
- Bật Open vSwitch.

In [ ]:
# Thêm user hiện tại vào group libvirt và kvm
sudo usermod -aG libvirt,kvm $USER

# Enable libvirtd
sudo systemctl enable --now libvirtd

> Lưu ý: Sau khi thêm user vào group `libvirt,kvm`, nên logout/login lại SSH để quyền group có hiệu lực.

In [ ]:
# XÓA default network libvirt để tránh conflict NAT
# Nếu báo network không tồn tại thì có thể bỏ qua.
virsh net-destroy default || true
virsh net-undefine default || true

# Xác nhận danh sách network libvirt
virsh net-list --all

Kết quả mong muốn: bảng `virsh net-list --all` trống hoặc không còn network `default`.

In [ ]:
# Enable Open vSwitch
sudo systemctl enable --now openvswitch-switch

# Kiểm tra OVS
sudo ovs-vsctl show

Kết quả mong muốn: lệnh `ovs-vsctl show` chạy không lỗi và có dòng `ovs_version`.

## BƯỚC 4 — Bật Nested KVM

Mục tiêu:
- Cho phép VM bên trong host vật lý vẫn có thể dùng ảo hóa phần cứng.
- Cần thiết nếu sau này chạy nested virtualization hoặc tạo VM bên trong môi trường lab.

In [ ]:
# Kiểm tra trạng thái Nested KVM hiện tại trên Intel CPU
cat /sys/module/kvm_intel/parameters/nested

Nếu kết quả là `N` hoặc `0` thì cần bật Nested KVM.
Nếu kết quả là `Y` hoặc `1` thì Nested KVM đã bật.

In [ ]:
# Bật Nested KVM cho Intel CPU
echo "options kvm-intel nested=1" | sudo tee /etc/modprobe.d/kvm-nested.conf

# Áp dụng ngay nếu chưa có VM nào đang chạy
sudo modprobe -r kvm_intel && sudo modprobe kvm_intel

# Xác nhận
cat /sys/module/kvm_intel/parameters/nested

Kết quả mong muốn: `Y`.

Nếu server dùng AMD CPU, thay `kvm_intel` bằng `kvm_amd`.

In [ ]:
# Dành cho AMD CPU nếu cần dùng
# cat /sys/module/kvm_amd/parameters/nested
# echo "options kvm-amd nested=1" | sudo tee /etc/modprobe.d/kvm-nested.conf
# sudo modprobe -r kvm_amd && sudo modprobe kvm_amd
# cat /sys/module/kvm_amd/parameters/nested

## BƯỚC 5 — Bật IOMMU

Mục tiêu:
- Chuẩn bị cho các bài lab nâng cao như PCI passthrough / GPU passthrough.
- Với Intel dùng `intel_iommu=on iommu=pt`.
- Với AMD dùng `amd_iommu=on iommu=pt`.

> Bước này cần sửa GRUB và reboot.

In [ ]:
# Mở file cấu hình GRUB
sudo nano /etc/default/grub

Trong file `/etc/default/grub`, tìm dòng:

```bash
GRUB_CMDLINE_LINUX=""
```

Nếu dùng Intel CPU, sửa thành:

```bash
GRUB_CMDLINE_LINUX="intel_iommu=on iommu=pt"
```

Nếu dùng AMD CPU, sửa thành:

```bash
GRUB_CMDLINE_LINUX="amd_iommu=on iommu=pt"
```

In [ ]:
# Cập nhật GRUB và reboot
sudo update-grub
sudo reboot

Sau khi reboot xong, SSH lại vào server và kiểm tra IOMMU.

In [ ]:
# Kiểm tra Intel IOMMU
dmesg | grep "Intel-IOMMU" | head -3

# Kiểm tra số lượng IOMMU group
ls /sys/kernel/iommu_groups/ | wc -l

# Kiểm tra Nested KVM vẫn còn bật sau reboot
cat /sys/module/kvm_intel/parameters/nested

Kết quả mong muốn:
- Có dòng `Intel-IOMMU: enabled`.
- Số lượng IOMMU group > 0.
- Nested KVM vẫn là `Y`.

## BƯỚC 6 — Chuẩn bị LVM Thin Pool

Đây là bước nguy hiểm nhất vì có thể xóa nhầm dữ liệu.

Mục tiêu:
- Tạo LVM Thin Pool để lưu VM local.
- Đăng ký pool này với libvirt dưới tên `vm-pool`.
- Chuẩn bị phân vùng riêng cho Ceph OSD.

## Nguyên tắc bắt buộc

Trước khi chạy `pvcreate`, `wipefs`, `fdisk`, phải xác định đúng disk/phân vùng.

- Disk hoặc partition đang chứa `/` là OS disk/partition.
- Tuyệt đối không dùng disk/partition đang chứa hệ điều hành.
- Nếu sai, có thể mất dữ liệu hoặc hỏng OS.

In [ ]:
# Xem disk vật lý
lsblk -d -o NAME,SIZE,TYPE,MOUNTPOINT

# Xem đầy đủ disk, partition, mountpoint
lsblk -o NAME,SIZE,TYPE,FSTYPE,MOUNTPOINT

## Trường hợp 1 — Có disk data riêng, ví dụ `/dev/sdb`

Ví dụ:
- `/dev/sda` là OS disk.
- `/dev/sdb` là data disk trống.
- Dùng `/dev/sdb` để tạo LVM Thin Pool.

In [ ]:
# Ví dụ dùng /dev/sdb làm LVM pool
# CHỈ CHẠY nếu chắc chắn /dev/sdb là disk data trống

sudo pvcreate /dev/sdb
sudo pvs

sudo vgcreate vg-lab /dev/sdb
sudo vgs

sudo lvcreate   --type thin-pool   --extents 95%VG   --name thin-pool   vg-lab

sudo lvs

Kết quả mong muốn khi chạy `sudo lvs`:
- Có logical volume tên `thin-pool`.
- Cột `Attr` có dạng gần giống `twi-a-tz--`.

## Trường hợp 2 — Chỉ có 1 disk lớn, cần chia partition

Ví dụ disk 1.1TB:
- Một phần cho OS.
- Một phần cho LVM pool, ví dụ `/dev/sda4`.
- Phần còn lại cho Ceph OSD, ví dụ `/dev/sda7`.

> Chỉ làm bước này nếu bạn hiểu rõ layout phân vùng hiện tại.

In [ ]:
# Kiểm tra layout trước khi chia
lsblk -o NAME,SIZE,TYPE,FSTYPE,MOUNTPOINT
sudo fdisk -l /dev/sda

Nếu cần tạo phân vùng `sda7` từ phần trống còn lại:

Trong `fdisk`:
- `n` → tạo partition mới
- `p` → primary nếu còn slot, hoặc làm theo gợi ý của fdisk
- nhập số partition: `7`
- Enter để dùng sector đầu mặc định
- Enter để dùng sector cuối mặc định hoặc nhập size mong muốn
- `w` để ghi thay đổi

Không chạy nếu chưa chắc chắn.

In [ ]:
# Tạo phân vùng mới từ vùng trống còn lại
sudo fdisk /dev/sda

# Reload partition table
sudo partprobe /dev/sda

# Kiểm tra lại
lsblk -o NAME,SIZE,TYPE,FSTYPE,MOUNTPOINT

## Tạo LVM Thin Pool từ partition, ví dụ `/dev/sda4`

In [ ]:
# CHỈ CHẠY nếu chắc chắn /dev/sda4 là partition dành cho LVM pool

sudo pvcreate /dev/sda4
sudo pvs

sudo vgcreate vg-lab /dev/sda4
sudo vgs

sudo lvcreate   --type thin-pool   --extents 95%VG   --name thin-pool   vg-lab

sudo lvs

## Đăng ký LVM Thin Pool với libvirt

Sau khi đã có VG `vg-lab` và thin pool `thin-pool`, đăng ký pool với libvirt dưới tên `vm-pool`.

In [ ]:
cat > /tmp/pool-vg-lab.xml << EOF
<pool type="logical">
  <name>vm-pool</name>
  <source>
    <name>vg-lab</name>
    <format type="lvm2"/>
  </source>
  <target>
    <path>/dev/vg-lab</path>
  </target>
</pool>
EOF

virsh pool-define /tmp/pool-vg-lab.xml
virsh pool-start vm-pool
virsh pool-autostart vm-pool

virsh pool-list --all

Kết quả mong muốn:

```text
Name      State    Autostart
vm-pool   active   yes
```

## Chuẩn bị Ceph OSD trên partition riêng, ví dụ `/dev/sda7`

Mục tiêu:
- Xóa signature cũ trên partition OSD.
- Add OSD vào Ceph cluster.

> Lưu ý: Lệnh `ceph orch daemon add osd ...` thường chạy từ node có quyền quản trị Ceph, ví dụ Server A hoặc node bootstrap Ceph.

In [ ]:
# CHỈ CHẠY nếu chắc chắn /dev/sda7 là partition dành riêng cho Ceph OSD

sudo wipefs -a /dev/sda7

Add OSD cho 3 server.

Cần thay đúng hostname thực tế:
- `server-a`
- `server-b`
- `server-c`

Nếu hostname thực tế là `stack1`, `stack2`, `stack3` hoặc tên khác thì phải thay lại.

In [ ]:
# Chạy trên node quản trị Ceph / Server A
sudo ceph orch daemon add osd server-a:/dev/sda7
sudo ceph orch daemon add osd server-b:/dev/sda7
sudo ceph orch daemon add osd server-c:/dev/sda7

## Kiểm tra sau khi thêm OSD

In [ ]:
ceph -s
ceph osd tree
ceph orch ps --daemon_type osd

Kết quả mong muốn:
- `ceph -s` không có lỗi nghiêm trọng.
- OSD mới xuất hiện trong `ceph osd tree`.
- OSD ở trạng thái `up/in`.

# Checklist hoàn thành Phần I

Đánh dấu sau khi hoàn thành trên từng server.

| Hạng mục | Server A | Server B | Server C |
|---|---|---|---|
| Update OS + reboot | ☐ | ☐ | ☐ |
| Cài package nền tảng | ☐ | ☐ | ☐ |
| User thuộc group libvirt,kvm | ☐ | ☐ | ☐ |
| libvirtd running | ☐ | ☐ | ☐ |
| default libvirt network đã xóa | ☐ | ☐ | ☐ |
| OVS running | ☐ | ☐ | ☐ |
| Nested KVM = Y | ☐ | ☐ | ☐ |
| IOMMU enabled | ☐ | ☐ | ☐ |
| LVM VG `vg-lab` tồn tại | ☐ | ☐ | ☐ |
| Libvirt pool `vm-pool` active/autostart | ☐ | ☐ | ☐ |
| Ceph OSD partition đã wipefs đúng | ☐ | ☐ | ☐ |
| OSD up/in | ☐ | ☐ | ☐ |

# Ghi chú vận hành

- Nên test toàn bộ quy trình trên Server A trước.
- Khi đã ổn, có thể dùng Ansible để chạy song song trên Server B và C.
- Riêng các thao tác chia disk/phân vùng nên kiểm tra thủ công từng server, không nên chạy mù bằng Ansible.
- Mọi lệnh có `/dev/sdX`, `/dev/sda4`, `/dev/sda7`, `/dev/sdb` đều phải thay đúng theo thực tế.